In [10]:
import copy
from collections import deque

# ---------------------------
# Parse Sudoku board from text
# ---------------------------
def parse_board(text: str) -> list[list[int]]:
    grid = []
    for line in text.strip().splitlines():
        line = line.strip()
        if line:
            # Convert each character into integer
            row = [int(ch) for ch in line]
            grid.append(row)
    # Ensure grid is 9x9
    assert len(grid) == 9 and all(len(r) == 9 for r in grid), "Board must be 9×9"
    return grid


# ---------------------------
# Create variable name for each cell
# ---------------------------
def cell(r: int, c: int) -> str:
    return f"{r},{c}"


# ---------------------------
# Build CSP (domains + neighbors)
# ---------------------------
def build_constraints(grid: list[list[int]]) -> tuple[dict, dict]:
    all_values = set(range(1, 10))

    # Domains: possible values for each cell
    domains: dict[str, set] = {}
    for r in range(9):
        for c in range(9):
            var = cell(r, c)
            if grid[r][c] != 0:
                # Fixed value
                domains[var] = {grid[r][c]}
            else:
                # Empty cell → all values possible
                domains[var] = set(all_values)

    # Neighbors: cells in same row, column, or box
    neighbors: dict[str, set] = {cell(r, c): set() for r in range(9) for c in range(9)}

    for r in range(9):
        for c in range(9):
            var = cell(r, c)

            # Row neighbors
            for cc in range(9):
                if cc != c:
                    neighbors[var].add(cell(r, cc))

            # Column neighbors
            for rr in range(9):
                if rr != r:
                    neighbors[var].add(cell(rr, c))

            # 3x3 box neighbors
            box_r, box_c = 3 * (r // 3), 3 * (c // 3)
            for dr in range(3):
                for dc in range(3):
                    nr, nc = box_r + dr, box_c + dc
                    if (nr, nc) != (r, c):
                        neighbors[var].add(cell(nr, nc))

    return domains, neighbors


# ---------------------------
# AC-3 Algorithm (Arc Consistency)
# ---------------------------
def ac3(domains: dict, neighbors: dict) -> bool:
    queue: deque[tuple[str, str]] = deque()

    # Add all arcs to queue
    for var in domains:
        for neighbor in neighbors[var]:
            queue.append((var, neighbor))

    while queue:
        xi, xj = queue.popleft()

        # Revise domain
        if revise(domains, xi, xj):
            # If domain becomes empty → failure
            if not domains[xi]:
                return False

            # Add affected arcs back to queue
            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True


# ---------------------------
# Revise function for AC-3
# ---------------------------
def revise(domains: dict, xi: str, xj: str) -> bool:
    revised = False
    for val in set(domains[xi]):
        # Remove value if no support in neighbor
        if domains[xj] == {val}:
            domains[xi].discard(val)
            revised = True
    return revised


# ---------------------------
# Forward Checking
# ---------------------------
def forward_check(domains: dict, var: str, value: int,
                  neighbors: dict) -> dict | None:
    pruned: dict[str, set] = {}
    for neighbor in neighbors[var]:
        if value in domains[neighbor]:
            domains[neighbor].discard(value)
            pruned.setdefault(neighbor, set()).add(value)
            # Failure if domain becomes empty
            if not domains[neighbor]:
                restore_pruned(domains, pruned)
                return None
    return pruned


# Restore removed values after backtracking
def restore_pruned(domains: dict, pruned: dict):
    for var, values in pruned.items():
        domains[var].update(values)


# ---------------------------
# MRV Heuristic (Minimum Remaining Values)
# ---------------------------
def select_unassigned_variable(assignment: dict, domains: dict) -> str | None:
    unassigned = [v for v in domains if v not in assignment]
    if not unassigned:
        return None
    return min(unassigned, key=lambda v: len(domains[v]))


# ---------------------------
# Order domain values
# ---------------------------
def order_domain_values(var: str, domains: dict) -> list:
    return sorted(domains[var])


# ---------------------------
# Stats class for tracking
# ---------------------------
class Stats:
    def __init__(self):
        self.calls = 0
        self.failures = 0


# ---------------------------
# Backtracking Search
# ---------------------------
def backtrack(assignment: dict, domains: dict, neighbors: dict,
              stats: Stats) -> dict | None:
    stats.calls += 1  # Count function calls

    # If all variables assigned → solution found
    if len(assignment) == 81:
        return assignment

    # Select variable using MRV
    var = select_unassigned_variable(assignment, domains)
    if var is None:
        return assignment

    # Try each value
    for value in order_domain_values(var, domains):
        if is_consistent(var, value, assignment, neighbors):
            assignment[var] = value

            # Copy domains for safe recursion
            domains_copy = {v: set(d) for v, d in domains.items()}
            domains_copy[var] = {value}

            # Apply forward checking
            pruned = forward_check(domains_copy, var, value, neighbors)
            if pruned is not None:
                result = backtrack(assignment, domains_copy, neighbors, stats)
                if result is not None:
                    return result

            # Undo assignment (backtrack)
            del assignment[var]

    stats.failures += 1  # Count failures
    return None


# ---------------------------
# Consistency check
# ---------------------------
def is_consistent(var: str, value: int, assignment: dict, neighbors: dict) -> bool:
    for neighbor in neighbors[var]:
        if assignment.get(neighbor) == value:
            return False
    return True


# ---------------------------
# Solve Sudoku
# ---------------------------
def solve(grid: list[list[int]]) -> tuple[list[list[int]] | None, Stats]:
    domains, neighbors = build_constraints(grid)

    # Apply AC-3 before search
    if not ac3(domains, neighbors):
        stats = Stats()
        stats.failures += 1
        return None, stats

    # Initialize assignment with given values
    assignment: dict[str, int] = {}
    for r in range(9):
        for c in range(9):
            if grid[r][c] != 0:
                assignment[cell(r, c)] = grid[r][c]

    stats = Stats()
    result = backtrack(assignment, domains, neighbors, stats)

    if result is None:
        return None, stats

    # Convert solution to grid format
    solved = [[0] * 9 for _ in range(9)]
    for r in range(9):
        for c in range(9):
            solved[r][c] = result[cell(r, c)]

    return solved, stats


# ---------------------------
# Display grid nicely
# ---------------------------
def grid_to_str(grid: list[list[int]]) -> str:
    lines = []
    for r, row in enumerate(grid):
        if r in (3, 6):
            lines.append("------+-------+------")
        parts = []
        for c, val in enumerate(row):
            if c in (3, 6):
                parts.append("|")
            parts.append(str(val) if val != 0 else ".")
        lines.append(" ".join(parts))
    return "\n".join(lines)


# ---------------------------
# Sudoku Boards
# ---------------------------
BOARDS = {
    "easy": """\
004030050
609400000
005100489
000060930
300807002
026040000
453009600
000004705
090050200""",

    "medium": """\
000030040
109700000
000851070
002607830
906010207
031502900
010369000
000005703
090070000""",

    "hard": """\
102040007
000080000
009500304
000607900
540000026
006405000
708003400
000010000
200060509""",

    "veryhard": """\
001007000
600400300
000030064
380076000
000000036
270015000
000020051
700100200
008090000""",
}


# ---------------------------
# Main function
# ---------------------------
def main():
    results = {}
    for name, board_text in BOARDS.items():

        print(f"  Board: {name.upper()}")

        grid = parse_board(board_text)
        print("\nInput:")
        print(grid_to_str(grid))

        solved, stats = solve(grid)

        if solved:
            print("\nSolution:")
            print(grid_to_str(solved))
        else:
            print("\n[NO SOLUTION FOUND]")

        print(f"\nBacktrack calls   : {stats.calls}")
        print(f"Backtrack failures: {stats.failures}")

        results[name] = {
            "grid": grid,
            "solved": solved,
            "calls": stats.calls,
            "failures": stats.failures,
        }

    return results


if __name__ == "__main__":
    main()

    

  Board: EASY

Input:
. . 4 | . 3 . | . 5 .
6 . 9 | 4 . . | . . .
. . 5 | 1 . . | 4 8 9
------+-------+------
. . . | . 6 . | 9 3 .
3 . . | 8 . 7 | . . 2
. 2 6 | . 4 . | . . .
------+-------+------
4 5 3 | . . 9 | 6 . .
. . . | . . 4 | 7 . 5
. 9 . | . 5 . | 2 . .

Solution:
7 8 4 | 9 3 2 | 1 5 6
6 1 9 | 4 8 5 | 3 2 7
2 3 5 | 1 7 6 | 4 8 9
------+-------+------
5 7 8 | 2 6 1 | 9 3 4
3 4 1 | 8 9 7 | 5 6 2
9 2 6 | 5 4 3 | 8 7 1
------+-------+------
4 5 3 | 7 2 9 | 6 1 8
8 6 2 | 3 1 4 | 7 9 5
1 9 7 | 6 5 8 | 2 4 3

Backtrack calls   : 50
Backtrack failures: 0
  Board: MEDIUM

Input:
. . . | . 3 . | . 4 .
1 . 9 | 7 . . | . . .
. . . | 8 5 1 | . 7 .
------+-------+------
. . 2 | 6 . 7 | 8 3 .
9 . 6 | . 1 . | 2 . 7
. 3 1 | 5 . 2 | 9 . .
------+-------+------
. 1 . | 3 6 9 | . . .
. . . | . . 5 | 7 . 3
. 9 . | . 7 . | . . .

Solution:
8 7 5 | 9 3 6 | 1 4 2
1 6 9 | 7 2 4 | 3 8 5
2 4 3 | 8 5 1 | 6 7 9
------+-------+------
4 5 2 | 6 9 7 | 8 3 1
9 8 6 | 4 1 3 | 2 5 7
7 3 1 | 5 8 2 | 9 6 4
------